### Tasks:
1. Calculate **IQR bounds** for `transaction_amount` and identify outlier indices.
2. Calculate **Z-Score** values and flag extreme values ($|Z| > 3.0$).
3. Fit an **Isolation Forest** to identify multi-variable anomalies across `transaction_amount` and `risk_score`.
4. Apply **Winsorization (capping)** to restrict outliers within IQR boundaries.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

np.random.seed(101)

# Sample Data Generation
data_size = 500
tx_amounts = np.random.exponential(scale=50, size=data_size)
risk_scores = np.random.normal(loc=50, scale=10, size=data_size)

# Inject anomalies
tx_amounts[:15] = np.random.uniform(300, 800, size=15)
risk_scores[:15] = np.random.uniform(90, 100, size=15)

df = pd.DataFrame({
    'transaction_amount': tx_amounts,
    'risk_score': risk_scores
})
df.head()

,transaction_amount,risk_score
0,735.961428,98.005781
1,356.679957,94.201870
2,582.949562,90.045716
3,634.725394,90.151992
4,356.890655,91.551146


### Task 1: Complete the IQR Outlier Function
Fill in the missing mathematical logic for calculating $Q1$, $Q3$, $IQR$, and the lower/upper bounds.

In [2]:
def get_iqr_outliers(series: pd.Series, k: float = 1.5):
    """
    Calculate IQR bounds and identify outlier indices.
    Lower Bound = Q1 - k * IQR
    Upper Bound = Q3 + k * IQR
    """
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    
    outlier_mask = (series < lower_bound) | (series > upper_bound)
    return outlier_mask, lower_bound, upper_bound

# Test your function
mask, lb, ub = get_iqr_outliers(df['transaction_amount'])
print(f"Q1 / Q3 IQR Bounds: [{lb:.2f}, {ub:.2f}]")
print(f"Detected {mask.sum()} outliers using IQR.")
print("Sample outlier rows:")
print(df[mask].head())


### Task 2: Isolation Forest Detection
Fit an `IsolationForest` estimator with `contamination=0.03` on both features (`transaction_amount` and `risk_score`).

In [3]:
# Task 2: Isolation Forest Detection
# Fit IsolationForest with contamination=0.03 across both continuous features
iso = IsolationForest(contamination=0.03, random_state=101)
iso.fit(df[['transaction_amount', 'risk_score']])

# Predict: -1 indicates anomaly, 1 indicates inlier
preds = iso.predict(df[['transaction_amount', 'risk_score']])
df['is_outlier_iforest'] = np.where(preds == -1, 1, 0)

print(f"Isolation Forest flagged {df['is_outlier_iforest'].sum()} anomalies.")
print(df[df['is_outlier_iforest'] == 1].head(10))


### Task 3: Z-Score Outlier Detection
Calculate Z-Score values ($Z = \frac{X - \mu}{\sigma}$) for `transaction_amount` and flag extreme values ($|Z| > 3.0$).

In [ ]:
# Task 3: Z-Score Outlier Detection
mean_val = df['transaction_amount'].mean()
std_val = df['transaction_amount'].std()

# Calculate Z-Score
df['z_score_tx'] = (df['transaction_amount'] - mean_val) / std_val
df['is_outlier_zscore'] = np.where(df['z_score_tx'].abs() > 3.0, 1, 0)

print(f"Detected {df['is_outlier_zscore'].sum()} outliers using Z-Score (|Z| > 3.0).")
print("Sample Z-Score outliers:")
print(df[df['is_outlier_zscore'] == 1][['transaction_amount', 'z_score_tx']].head())


### Task 4: Winsorization (IQR-based Capping)
Apply Winsorization (capping) to restrict outliers in `transaction_amount` within the calculated IQR boundaries.

In [ ]:
# Task 4: Winsorization (Capping) to IQR Bounds
_, lower_bound, upper_bound = get_iqr_outliers(df['transaction_amount'])

# Cap values below lower_bound to lower_bound, and above upper_bound to upper_bound
df['transaction_amount_winsorized'] = np.clip(df['transaction_amount'], lower_bound, upper_bound)

print("--- Summary Before vs After Winsorization ---")
print("Original transaction_amount:")
print(df['transaction_amount'].describe().round(2))
print("\nWinsorized transaction_amount:")
print(df['transaction_amount_winsorized'].describe().round(2))
